# TP-MCTS Experiments

Two experiment scripts, run from this notebook.

| Script | Purpose |
|--------|---------|
| `run_mcts_heuristic_comparison.py` | TP-MCTS score across 4 NASA scenarios × 5 heuristics |
| `run_heuristic_runtime_per_call.py` | Per-call timing (wrapper + worker + cache hit/miss) |

**Setup:** clone the repo and `cd` into it first (same as `demo.ipynb` cells 1-4).

In [2]:
# Clone repo (skip if already done in this Colab session)
import os

if not os.path.exists('/content/tp_mcts'):
    %cd /content
    !git clone https://github.com/eliezerRevach/tp_mcts.git

%cd /content/tp_mcts
!pip -q install dill numpy pandas
print('Ready:', os.getcwd())

/content
Cloning into 'tp_mcts'...
remote: Enumerating objects: 2558, done.
remote: Counting objects: 100% (587/587), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 2558 (delta 533), reused 534 (delta 497), pack-reused 1971 (from 1)
Receiving objects: 100% (2558/2558), 15.61 MiB | 7.71 MiB/s, done.
Resolving deltas: 100% (1553/1553), done.
Updating files: 100% (180/180), done.
/content/tp_mcts
Ready: /content/tp_mcts


## Config

Edit the values below, then run the cells for each experiment.

In [3]:
# ── Shared config ─────────────────────────────────────────────────────────
from pathlib import Path

def _find_repo_root() -> Path:
    """Locate repo root (folder containing scripts/run_mcts_heuristic_comparison.py)."""
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "run_mcts_heuristic_comparison.py").is_file():
            return p
    return start

REPO_ROOT = _find_repo_root()

RUNS                 = 20     # runs per (scenario × heuristic)
SEED                 = 123    # random seed — same seed across all runs
SEARCH_TIME          = 1      # MCTS search time per step (seconds)
EXPLORATION_CONSTANT = 0.05  # UCT exploration constant C
REWARD_MODE          = "deadline"  # "deadline" or "terminal"

# Scenario grid — Script 1 only
OBJECTS      = [2, 3]      # object_amount values
DEADLINES    = [25, 35]    # deadline values

# Heuristics — used by both scripts
# Options: ptrpg_old  baseline  baseline_cached  atomic_exact  atomic_exact_cached
HEURISTICS   = ["ptrpg_old", "baseline", "baseline_cached", "atomic_exact", "atomic_exact_cached"]

# Runtime benchmark settings — Script 2 only
RT_OBJECTS   = 2
RT_DEADLINE  = 35
RT_H_DEPTH   = 35
RT_MAX_STEPS = 1000

# Output paths — absolute so download / pandas always match where scripts write
MCTS_CSV    = str((REPO_ROOT / "results" / "mcts_heuristic_comparison.csv").resolve())
RUNTIME_CSV = str((REPO_ROOT / "results" / "heuristic_runtime_per_call.csv").resolve())

print("Config:")
print(f"  REPO_ROOT      : {REPO_ROOT}")
print(f"  MCTS scenarios : objects={OBJECTS}  deadlines={DEADLINES}  runs={RUNS}  seed={SEED}  C={EXPLORATION_CONSTANT}")
print(f"  Heuristics     : {HEURISTICS}")
print(f"  Runtime bench  : nasa_rover obj={RT_OBJECTS}  deadline={RT_DEADLINE}  depth={RT_H_DEPTH}")
print(f"  MCTS_CSV       : {MCTS_CSV}")
print(f"  RUNTIME_CSV    : {RUNTIME_CSV}")

Config:
  REPO_ROOT      : /content/tp_mcts
  MCTS scenarios : objects=[2, 3]  deadlines=[25, 35]  runs=20  seed=123  C=0.05
  Heuristics     : ['ptrpg_old', 'baseline', 'baseline_cached', 'atomic_exact', 'atomic_exact_cached']
  Runtime bench  : nasa_rover obj=2  deadline=35  depth=35
  MCTS_CSV       : /content/tp_mcts/results/mcts_heuristic_comparison.csv
  RUNTIME_CSV    : /content/tp_mcts/results/heuristic_runtime_per_call.csv


## Script 1 — MCTS Heuristic Comparison

Runs TP-MCTS on each **(object_amount, deadline) × heuristic** combination.

- Results are saved **incrementally** — partial data survives a Colab timeout.
- Output: `results/mcts_heuristic_comparison.csv`

In [ ]:
import subprocess

_h   = " ".join(HEURISTICS)
_obj = " ".join(str(o) for o in OBJECTS)
_dl  = " ".join(str(d) for d in DEADLINES)

cmd = [
    "python", "scripts/run_mcts_heuristic_comparison.py",
    "--runs",                 str(RUNS),
    "--seed",                 str(SEED),
    "--search_time",          str(SEARCH_TIME),
    "--exploration_constant", str(EXPLORATION_CONSTANT),
    "--reward_mode",          REWARD_MODE,
    "--output",               MCTS_CSV,
    "--objects",              *[str(o) for o in OBJECTS],
    "--deadlines",            *[str(d) for d in DEADLINES],
    "--heuristics",           *HEURISTICS,
]

print("Running:", " ".join(cmd), flush=True)
print()

# Stream output live — cwd=REPO_ROOT so relative paths match this notebook
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(REPO_ROOT),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

Running: python scripts/run_mcts_heuristic_comparison.py --runs 20 --seed 123 --search_time 1 --exploration_constant 0.05 --reward_mode deadline --output /content/tp_mcts/results/mcts_heuristic_comparison.csv --objects 2 3 --deadlines 25 35 --heuristics ptrpg_old baseline baseline_cached atomic_exact atomic_exact_cached


  TP-MCTS Heuristic Comparison
  Domain    : nasa_rover
  Scenarios : [(2, 25), (2, 35), (3, 25), (3, 35)]
  Heuristics: ['ptrpg_old', 'baseline', 'baseline_cached', 'atomic_exact', 'atomic_exact_cached']
  Runs/exp  : 20  seed=123  C=0.05  reward_mode=deadline
  Total     : 20 experiments
  Output    : /content/tp_mcts/results/mcts_heuristic_comparison.csv

[1/20]

────────────────────────────────────────────────────────────
  [nasa_rover obj=2 dl=25] heuristic=ptrpg_old
  depth=25  runs=20  seed=123  C=0.05  reward_mode=deadline
────────────────────────────────────────────────────────────
  => success=8/20  rate=0.4  avg_time=22.75  wall=2012.5s
Wrote /content/tp_mc

: 

In [ ]:
import pandas as pd

df = pd.read_csv(MCTS_CSV)

# Pivot: rows = (objects, deadline), columns = heuristic, values = success_rate
pivot = df.pivot_table(
    index=["object_amount", "deadline"],
    columns="heuristic",
    values="success_rate",
    aggfunc="first",
)
print("=== Success rate by scenario × heuristic ===")
print(pivot.to_string())

print("\n=== Full results table ===")
cols = ["domain", "object_amount", "deadline", "heuristic",
        "amount_success", "success_rate", "avg_success_time", "std_success_time"]
print(df[cols].to_string(index=False))

## Script 2 — Heuristic Per-Call Runtime Benchmark

Runs `greedy_parallel` on one scenario for each heuristic and measures:

- `wrapper_avg_call_sec` — total heuristic call cost (includes STN work)
- `worker_avg_call_sec` — pure propagation cost
- `worker_cache_hit_avg_sec` / `worker_cache_miss_avg_sec` — cache breakdown

Output: `results/heuristic_runtime_per_call.csv`

In [4]:
import subprocess

cmd = [
    "python", "scripts/run_heuristic_runtime_per_call.py",
    "--domain",         "nasa_rover",
    "--object_amount",  str(RT_OBJECTS),
    "--deadline",       str(RT_DEADLINE),
    "--heuristic_depth",str(RT_H_DEPTH),
    "--max_steps",      str(RT_MAX_STEPS),
    "--seed",           str(SEED),
    "--output",         RUNTIME_CSV,
    "--heuristics",     *HEURISTICS,
]

print("Running:", " ".join(cmd), flush=True)
print()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(REPO_ROOT),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

Running: python scripts/run_heuristic_runtime_per_call.py --domain nasa_rover --object_amount 2 --deadline 35 --heuristic_depth 35 --max_steps 1000 --seed 123 --output /content/tp_mcts/results/heuristic_runtime_per_call.csv --heuristics ptrpg_old baseline baseline_cached atomic_exact atomic_exact_cached


  Heuristic Per-Call Runtime Benchmark
  Scenario  : nasa_rover  obj=2  deadline=35
  H-depth   : 35  max_steps=1000  seed=123
  Heuristics: ['ptrpg_old', 'baseline', 'baseline_cached', 'atomic_exact', 'atomic_exact_cached']
  Output    : /content/tp_mcts/results/heuristic_runtime_per_call.csv

  Heuristic : ptrpg_old  (ptrpg_old (trpg))
  Internal  : heuristic_name=trpg  strategy=baseline
  Scenario  : nasa_rover obj=2  deadline=35  depth=35
started step 0
Current state is state: store_of(s0, r0) ; store_of(s1, r0) ; on_board(c0, r0) ; hand_of(h0, r0) ; hand_of(h1, r0) ; store_of(s2, r1) ; store_of(s3, r1) ; on_board(c1, r1) ; hand_of(h2, r1) ; hand_of(h3, r1) ; free_h(h0) ; good(h0)

In [5]:
import pandas as pd

df_rt = pd.read_csv(RUNTIME_CSV)

timing_cols = [
    "heuristic",
    "wrapper_avg_call_sec",
    "worker_avg_call_sec",
    "worker_cache_hit_avg_sec",
    "worker_cache_miss_avg_sec",
    "worker_cache_hits",
    "worker_cache_misses",
    "plan_success",
]

df_sorted = df_rt[timing_cols].sort_values("wrapper_avg_call_sec")

print("=== Per-call runtime ranking (fastest → slowest) ===")
print(df_sorted.to_string(index=False))

=== Per-call runtime ranking (fastest → slowest) ===
          heuristic  wrapper_avg_call_sec  worker_avg_call_sec  worker_cache_hit_avg_sec  worker_cache_miss_avg_sec  worker_cache_hits  worker_cache_misses  plan_success
          ptrpg_old              0.027140                  NaN                       NaN                        NaN                  0                    0          True
       atomic_exact              0.034783             0.034720                  0.000026                   0.039841                 84                  569          True
atomic_exact_cached              0.035322             0.035259                  0.000027                   0.040460                 84                  569          True
    baseline_cached              0.037292             0.037228                  0.000032                   0.042720                 84                  569          True
           baseline              0.072214             0.072102                  0.000037         

## Download Results to your PC

Run this cell after any experiment.

- **On Colab**: Colab cannot write to a path on your PC. This cell triggers a **browser download** (usually to **Downloads**). To land files in `TP_MCTS\results` on Windows, either move them after download, or set your browser’s default download folder to that directory (Chrome: Settings → Downloads → Location).
- **Local (this repo on your machine)**: copies each CSV into `LOCAL_PC_RESULTS_DIR` (default: `C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results`). Override with env var `TP_MCTS_RESULTS_DIR`.

Re-run the **Config** cell first so `MCTS_CSV` / `RUNTIME_CSV` are absolute paths under `REPO_ROOT`. If Script 1 was never run in this session, the MCTS CSV is skipped until you run it.

In [14]:
from pathlib import Path
import os
import shutil


def _find_repo_root_dl() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "run_mcts_heuristic_comparison.py").is_file():
            return p
    return start


# Local copy destination (not Colab). Override with env TP_MCTS_RESULTS_DIR.
if "TP_MCTS_RESULTS_DIR" in os.environ:
    LOCAL_PC_RESULTS_DIR = Path(os.environ["TP_MCTS_RESULTS_DIR"])
elif os.name == "nt":
    LOCAL_PC_RESULTS_DIR = Path(r"C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results")
else:
    LOCAL_PC_RESULTS_DIR = _find_repo_root_dl() / "results"
# Hint on Colab (Linux VM): where to put files on your PC; override with TP_MCTS_RESULTS_DIR.
_PC_RESULTS_HINT = (
    Path(os.environ["TP_MCTS_RESULTS_DIR"])
    if "TP_MCTS_RESULTS_DIR" in os.environ
    else Path(r"C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results")
)


def _resolve_csv(path: str) -> Path | None:
    """Resolve CSV path even if kernel cwd != repo root (Colab / multi-root)."""
    p = Path(path)
    if p.is_file():
        return p
    alt = _find_repo_root_dl() / "results" / p.name
    if alt.is_file():
        return alt
    return None


def _is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def download_result(path: str) -> None:
    """Colab: browser download. Local: copy into LOCAL_PC_RESULTS_DIR."""
    p = _resolve_csv(path)
    if p is None:
        print(f"  [skip] not found: {path}")
        print(f"          tried: {Path(path).resolve()} and {_find_repo_root_dl() / 'results' / Path(path).name}")
        return
    sp = str(p.resolve())
    size_kb = p.stat().st_size / 1024

    if _is_colab():
        from google.colab import files

        files.download(sp)
        print(f"  [browser download] {sp}  ({size_kb:.1f} KB)")
        print(f"      → On your PC, move the file to: {_PC_RESULTS_HINT}")
        print("      (Or set Chrome/Edge default download folder to that path.)")
        return

    LOCAL_PC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    dest = LOCAL_PC_RESULTS_DIR / p.name
    shutil.copy2(p, dest)
    print(f"  [copied] {sp}  ({size_kb:.1f} KB)")
    print(f"      → {dest.resolve()}")


print("Exporting results...")
download_result(MCTS_CSV)
download_result(RUNTIME_CSV)
print("Done.")

Exporting results...
  [skip] not found: /content/tp_mcts/results/mcts_heuristic_comparison.csv
          tried: /content/tp_mcts/results/mcts_heuristic_comparison.csv and /content/tp_mcts/results/mcts_heuristic_comparison.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  [browser download] /content/tp_mcts/results/heuristic_runtime_per_call.csv  (1.3 KB)
      → On your PC, move the file to: C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results
      (Or set Chrome/Edge default download folder to that path.)
Done.


### If you cannot find results (Colab)

- **`/content` is erased** when the runtime restarts or disconnects. If you open Colab later without re-running the experiment cells, **`tp_mcts/results` will be empty** — there is nothing to download.
- **Downloads**: the file may be blocked, or saved under another browser profile / “Ask where to save” folder.

**Run the next cell** in the same session **after** experiments: it checks whether the CSVs exist, lists `results/`, and **shows the tables inside the notebook** (works even when download fails).

**To keep files across sessions**, mount Google Drive and copy `results/` there (see comments in that cell), or download from the **Files** sidebar: `content` → `tp_mcts` → `results` → right‑click → **Download**.

In [15]:
# Locate results + show CSVs in the notebook (works on Colab without using Downloads).
from pathlib import Path

try:
    _rr = REPO_ROOT
except NameError:
    raise RuntimeError("Run the **Config** cell first (defines REPO_ROOT, MCTS_CSV, RUNTIME_CSV).") from None

results_dir = Path(REPO_ROOT) / "results"
print(f"REPO_ROOT     : {REPO_ROOT}")
print(f"results folder: {results_dir}  (exists={results_dir.is_dir()})")
print()

for label, path_str in [("MCTS_CSV", MCTS_CSV), ("RUNTIME_CSV", RUNTIME_CSV)]:
    p = Path(path_str)
    st = "OK " if p.is_file() else "MISSING — run the experiment script cell in this session"
    print(f"{st}  {label}: {p}")

if results_dir.is_dir():
    print("\nFiles in results/:")
    for f in sorted(results_dir.iterdir()):
        if f.is_file():
            print(f"  {f.name}  ({f.stat().st_size} bytes)")

import pandas as pd
from IPython.display import display

for label, path_str in [("MCTS comparison", MCTS_CSV), ("Heuristic runtime", RUNTIME_CSV)]:
    p = Path(path_str)
    if p.is_file():
        df = pd.read_csv(p)
        print(f"\n=== {label}: {p.name} ({len(df)} rows) ===")
        display(df)
    else:
        print(f"\n=== {label}: skip (file not found) ===")

# --- Optional: persist on Google Drive (uncomment, run mount cell first) ---
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# dest = Path("/content/drive/MyDrive/Colab_TP_MCTS_results")
# dest.mkdir(parents=True, exist_ok=True)
# for path_str in [MCTS_CSV, RUNTIME_CSV]:
#     p = Path(path_str)
#     if p.is_file():
#         shutil.copy2(p, dest / p.name)
#         print("Copied to Drive:", dest / p.name)

REPO_ROOT     : /content/tp_mcts
results folder: /content/tp_mcts/results  (exists=True)

MISSING — run the experiment script cell in this session  MCTS_CSV: /content/tp_mcts/results/mcts_heuristic_comparison.csv
OK   RUNTIME_CSV: /content/tp_mcts/results/heuristic_runtime_per_call.csv

Files in results/:
  heuristic_runtime_per_call.csv  (1315 bytes)

=== MCTS comparison: skip (file not found) ===

=== Heuristic runtime: heuristic_runtime_per_call.csv (5 rows) ===


,heuristic,heuristic_label,heuristic_name_internal,strategy_internal,domain,object_amount,deadline,seed,heuristic_depth,max_steps,...,wrapper_first_call_sec,wrapper_avg_call_sec,worker_total_calls,worker_total_time_sec,worker_first_call_sec,worker_avg_call_sec,worker_cache_hits,worker_cache_misses,worker_cache_hit_avg_sec,worker_cache_miss_avg_sec
0,ptrpg_old,ptrpg_old (trpg),trpg,NaN,nasa_rover,2,35,123,35,1000,...,0.032591,0.027140,0,NaN,NaN,NaN,0,0,NaN,NaN
1,baseline,baseline,temporal_probabilistic_rpg,baseline,nasa_rover,2,35,123,35,1000,...,0.178845,0.072214,653,47.082740,0.119599,0.072102,84,569,0.000037,0.082741
2,baseline_cached,baseline_cached,temporal_probabilistic_rpg,baseline_cached,nasa_rover,2,35,123,35,1000,...,0.115166,0.037292,653,24.310135,0.087671,0.037228,84,569,0.000032,0.042720
3,atomic_exact,atomic_exact,temporal_probabilistic_rpg,atom_backtrack_exact,nasa_rover,2,35,123,35,1000,...,0.097760,0.034783,653,22.671865,0.069039,0.034720,84,569,0.000026,0.039841
4,atomic_exact_cached,atomic_exact_cached,temporal_probabilistic_rpg,atom_backtrack_cached,nasa_rover,2,35,123,35,1000,...,0.109087,0.035322,653,23.024110,0.080846,0.035259,84,569,0.000027,0.040460
